# PepsiCo Financial Statement Analysis: Data Extraction

Financial statement data for PepsiCo, Inc. (CIK 0000077476) is pulled from SEC EDGAR's XBRL company facts API. Each annual report repeats prior years for comparison and older reports include quarterly figures, so every line item has several candidate values per fiscal year. The cell below lists all candidates for net revenue to show what the extraction has to choose between.

In [1]:
import requests
import pandas as pd
from datetime import date

CIK = "0000077476"
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json"
headers = {"User-Agent": "Fatiha Okesola fatihaayink@gmail.com"}

data = requests.get(url, headers=headers).json()
gaap = data["facts"]["us-gaap"]

# Every annual-filing fact reported for revenue, one row each
rows = []
for e in gaap["Revenues"]["units"]["USD"]:
    if e["form"].startswith("10-K"):
        start = e.get("start")
        days = (date.fromisoformat(e["end"]) - date.fromisoformat(start)).days if start else None
        rows.append({
            "end": e["end"], "start": start, "days": days,
            "value_bn": round(e["val"] / 1e9, 2),
            "form": e["form"], "fy_label": e["fy"], "filed": e["filed"], "accn": e["accn"],
        })

facts = pd.DataFrame(rows).sort_values(["end", "filed"])
facts[facts["end"] >= "2016-01-01"]

,end,start,days,value_bn,form,fy_label,filed,accn
0,2016-12-31,2015-12-27,370,62.80,10-K,2018,2019-02-15,0000077476-19-000017
1,2017-12-30,2017-01-01,363,63.52,10-K,2018,2019-02-15,0000077476-19-000017
2,2017-12-30,2017-01-01,363,63.52,10-K,2019,2020-02-13,0000077476-20-000015
3,2018-03-24,2017-12-31,83,12.56,10-K,2019,2020-02-13,0000077476-20-000015
4,2018-06-16,2018-03-25,83,16.09,10-K,2019,2020-02-13,0000077476-20-000015
5,2018-09-08,2018-06-17,83,16.48,10-K,2019,2020-02-13,0000077476-20-000015
6,2018-12-29,2017-12-31,363,64.66,10-K,2018,2019-02-15,0000077476-19-000017
7,2018-12-29,2017-12-31,363,64.66,10-K,2019,2020-02-13,0000077476-20-000015
9,2018-12-29,2018-09-09,111,19.52,10-K,2019,2020-02-13,0000077476-20-000015
8,2018-12-29,2017-12-31,363,64.66,10-K,2020,2021-02-11,0000077476-21-000007


## Tag coverage

Each concept is tested against its candidate XBRL tags. Coverage counts only annual-filing facts and, for income statement and cash flow items, only full-year periods (longer than 350 days). PepsiCo's quarters run 83 days, or 111 days for the fourth quarter, so this threshold separates them cleanly from 52- and 53-week years.

In [2]:
def is_full_year(e):
    # Balance-sheet items have no start date; flow items must span roughly a year
    if "start" not in e:
        return True
    days = (date.fromisoformat(e["end"]) - date.fromisoformat(e["start"])).days
    return days > 350

candidates = {
    "Net revenue": ["Revenues", "SalesRevenueNet", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "Cost of sales": ["CostOfGoodsAndServicesSold", "CostOfGoodsSold", "CostOfRevenue"],
    "Gross profit": ["GrossProfit"],
    "Operating profit": ["OperatingIncomeLoss"],
    "Income before tax": ["IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
                          "IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments"],
    "Income tax": ["IncomeTaxExpenseBenefit"],
    "Net income (consolidated)": ["ProfitLoss"],
    "Net income (attributable)": ["NetIncomeLoss"],
    "Interest expense": ["InterestExpense", "InterestExpenseNonoperating", "InterestExpenseDebt"],
    "D&A": ["DepreciationDepletionAndAmortization", "DepreciationAmortizationAndAccretionNet", "DepreciationAndAmortization"],
    "Cash from operations": ["NetCashProvidedByUsedInOperatingActivities"],
    "Cash from investing": ["NetCashProvidedByUsedInInvestingActivities"],
    "Capital spending": ["PaymentsToAcquirePropertyPlantAndEquipment", "PaymentsToAcquireProductiveAssets"],
    "Cash": ["CashAndCashEquivalentsAtCarryingValue"],
    "Short-term investments": ["ShortTermInvestments"],
    "Current assets": ["AssetsCurrent"],
    "Current liabilities": ["LiabilitiesCurrent"],
    "Total assets": ["Assets"],
    "Total liabilities": ["Liabilities"],
    "PepsiCo equity": ["StockholdersEquity"],
    "Short-term debt": ["DebtCurrent", "ShortTermBorrowings", "LongTermDebtCurrent"],
    "Long-term debt": ["LongTermDebtNoncurrent"],
    "Payables (combined line)": ["AccountsPayableAndAccruedLiabilitiesCurrent"],
    "Trade payables": ["AccountsPayableCurrent"],
    "Supplier finance": ["SupplierFinanceProgramObligation", "SupplierFinanceProgramObligationCurrent"],
}

check = []
for concept, tags in candidates.items():
    for tag in tags:
        if tag not in gaap or "USD" not in gaap[tag]["units"]:
            check.append({"concept": concept, "tag": tag, "years": "NOT USED", "missing": ""})
            continue
        years = sorted({int(e["end"][:4]) for e in gaap[tag]["units"]["USD"]
                        if e["form"].startswith("10-K") and is_full_year(e)
                        and 2016 <= int(e["end"][:4]) <= 2025})
        missing = [y for y in range(2016, 2026) if y not in years]
        check.append({"concept": concept, "tag": tag,
                      "years": f"{years[0]}–{years[-1]}" if years else "none in range",
                      "missing": ", ".join(map(str, missing))})

pd.set_option("display.max_colwidth", None)
pd.DataFrame(check)

,concept,tag,years,missing
0,Net revenue,Revenues,2016–2025,
1,Net revenue,SalesRevenueNet,2016–2017,"2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025"
2,Net revenue,RevenueFromContractWithCustomerExcludingAssessedTax,NOT USED,
3,Cost of sales,CostOfGoodsAndServicesSold,2016–2025,
4,Cost of sales,CostOfGoodsSold,NOT USED,
5,Cost of sales,CostOfRevenue,NOT USED,
6,Gross profit,GrossProfit,2016–2025,
7,Operating profit,OperatingIncomeLoss,2016–2025,
8,Income before tax,IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest,2016–2025,
9,Income before tax,IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments,none in range,"2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025"


Revenue was tagged `SalesRevenueNet` in filings before 2019 and `Revenues` afterwards. Later filings repeat 2016 and 2017 under `Revenues`, so that tag alone covers the full period. Capital spending is reported under `PaymentsToAcquireProductiveAssets` and supplier finance obligations under `SupplierFinanceProgramObligationCurrent` from 2023, the first year of the disclosure.

## Extraction

One value is selected per concept per fiscal year: annual filings only, full-year periods for flow items, December period ends, and the most recently filed fact where a year is reported more than once, so any recast figure replaces the original. The accession number of the source filing is kept for every value.

In [3]:
selected = {
    "Net revenue": "Revenues",
    "Cost of sales": "CostOfGoodsAndServicesSold",
    "Gross profit": "GrossProfit",
    "Operating profit": "OperatingIncomeLoss",
    "Income before tax": "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
    "Income tax": "IncomeTaxExpenseBenefit",
    "Net income (consolidated)": "ProfitLoss",
    "Net income (attributable)": "NetIncomeLoss",
    "Interest expense": "InterestExpense",
    "D&A (option A)": "DepreciationDepletionAndAmortization",
    "D&A (option B)": "DepreciationAndAmortization",
    "Cash from operations": "NetCashProvidedByUsedInOperatingActivities",
    "Cash from investing": "NetCashProvidedByUsedInInvestingActivities",
    "Capital spending": "PaymentsToAcquireProductiveAssets",
    "Cash": "CashAndCashEquivalentsAtCarryingValue",
    "Short-term investments": "ShortTermInvestments",
    "Current assets": "AssetsCurrent",
    "Current liabilities": "LiabilitiesCurrent",
    "Total assets": "Assets",
    "Total liabilities": "Liabilities",
    "PepsiCo equity": "StockholdersEquity",
    "Short-term debt": "DebtCurrent",
    "Long-term debt": "LongTermDebtNoncurrent",
    "Payables (combined line)": "AccountsPayableAndAccruedLiabilitiesCurrent",
    "Trade payables": "AccountsPayableCurrent",
    "Supplier finance": "SupplierFinanceProgramObligationCurrent",
}

def extract(tag, first_year=2017, last_year=2025):
    """One value per fiscal year: annual filings, full years, December period ends,
    latest filing wins, accession number breaks ties."""
    best = {}
    for e in gaap[tag]["units"]["USD"]:
        year = int(e["end"][:4])
        if not (e["form"].startswith("10-K") and is_full_year(e)
                and e["end"][5:7] == "12" and first_year <= year <= last_year):
            continue
        if year not in best or (e["filed"], e["accn"]) > (best[year]["filed"], best[year]["accn"]):
            best[year] = e
    return best

values, sources = {}, {}
for concept, tag in selected.items():
    picked = extract(tag)
    values[concept] = {y: e["val"] / 1e6 for y, e in picked.items()}
    sources[concept] = {y: e["accn"] for y, e in picked.items()}

values_df = pd.DataFrame(values).T.sort_index(axis=1).round(0)
sources_df = pd.DataFrame(sources).T.sort_index(axis=1)
values_df

,2017,2018,2019,2020,2021,2022,2023,2024,2025
Net revenue,63525.0,64661.0,67161.0,70372.0,79474.0,86392.0,91471.0,91854.0,93925.0
Cost of sales,28796.0,29381.0,30132.0,31797.0,37075.0,40576.0,41881.0,41744.0,43066.0
Gross profit,34729.0,35280.0,37029.0,38575.0,42399.0,45816.0,49590.0,50110.0,50859.0
Operating profit,10276.0,10110.0,10291.0,10080.0,11162.0,11512.0,11986.0,12887.0,11498.0
Income before tax,9602.0,9189.0,9312.0,9069.0,9821.0,10705.0,11417.0,11946.0,10244.0
Income tax,4694.0,-3370.0,1959.0,1894.0,2142.0,1727.0,2262.0,2320.0,1949.0
Net income (consolidated),4908.0,12559.0,7353.0,7175.0,7679.0,8978.0,9155.0,9626.0,8295.0
Net income (attributable),4857.0,12515.0,7314.0,7120.0,7618.0,8910.0,9074.0,9578.0,8240.0
Interest expense,1151.0,1219.0,935.0,1128.0,1863.0,939.0,819.0,919.0,1121.0
D&A (option A),2369.0,2399.0,2432.0,2548.0,2710.0,2763.0,2948.0,3160.0,3451.0


## Final source mapping

Two concepts change after comparing the extracted values with the filings.

**Depreciation and amortisation.** `DepreciationDepletionAndAmortization` matches the cash flow statement line in every year (for example $2,432 million in 2019 and $3,451 million in 2025). `DepreciationAndAmortization` reports a smaller figure until 2022, so it is dropped.

**Interest expense.** The `InterestExpense` tag does not hold the same measure across years. For 2017 it carries gross interest expense ($1,151 million), the figure on the income statement at the time. From 2018 onwards it carries the net figure ("net interest expense and other", $1,121 million in 2025), because later filings present interest on a net basis and re-tag earlier years accordingly. Interest coverage needs gross interest expense, so it is taken from the filings instead, using the most recent 10-K that reports each year.

In [4]:
# Depreciation and amortisation: keep the tag that matches the cash flow statement
values_df = values_df.drop(index=["D&A (option B)", "Interest expense"]).rename(index={"D&A (option A)": "D&A"})
sources_df = sources_df.drop(index=["D&A (option B)", "Interest expense"]).rename(index={"D&A (option A)": "D&A"})

# Gross interest expense, $ millions, entered from the filings (filing-note source)
gross_interest = {
    2017: (1151, "2019 10-K, PDF p.75, income statement"),
    2018: (1525, "2019 10-K, PDF p.75, income statement"),
    2019: (1135, "2020 10-K, PDF p.53, ROIC table in MD&A"),
    2020: (1252, "2021 10-K, PDF p.54, ROIC table in MD&A"),
    2021: (1988, "2022 10-K, PDF p.56, ROIC table in MD&A"),
    2022: (1119, "2023 10-K, PDF p.56, ROIC table in MD&A"),
    2023: (1437, "2023 10-K, PDF p.56, ROIC table in MD&A"),
    2024: (1606, "2024 10-K, PDF p.57, ROIC table in MD&A"),
    2025: (1840, "2025 10-K, PDF p.54, ROIC table in MD&A"),
}
values_df.loc["Interest expense (gross)"] = pd.Series({y: v for y, (v, s) in gross_interest.items()})
sources_df.loc["Interest expense (gross)"] = pd.Series({y: "filing-note source: " + s for y, (v, s) in gross_interest.items()})
values_df

,2017,2018,2019,2020,2021,2022,2023,2024,2025
Net revenue,63525.0,64661.0,67161.0,70372.0,79474.0,86392.0,91471.0,91854.0,93925.0
Cost of sales,28796.0,29381.0,30132.0,31797.0,37075.0,40576.0,41881.0,41744.0,43066.0
Gross profit,34729.0,35280.0,37029.0,38575.0,42399.0,45816.0,49590.0,50110.0,50859.0
Operating profit,10276.0,10110.0,10291.0,10080.0,11162.0,11512.0,11986.0,12887.0,11498.0
Income before tax,9602.0,9189.0,9312.0,9069.0,9821.0,10705.0,11417.0,11946.0,10244.0
Income tax,4694.0,-3370.0,1959.0,1894.0,2142.0,1727.0,2262.0,2320.0,1949.0
Net income (consolidated),4908.0,12559.0,7353.0,7175.0,7679.0,8978.0,9155.0,9626.0,8295.0
Net income (attributable),4857.0,12515.0,7314.0,7120.0,7618.0,8910.0,9074.0,9578.0,8240.0
D&A,2369.0,2399.0,2432.0,2548.0,2710.0,2763.0,2948.0,3160.0,3451.0
Cash from operations,10030.0,9415.0,9649.0,10613.0,11616.0,10811.0,13442.0,12507.0,12087.0


## Validation against the 2025 10-K

Every extracted 2025 value is compared with the figure printed in the 2025 annual report. Capital spending appears as a negative number in the cash flow statement and is stored here as a positive amount.

In [5]:
filing_2025 = {
    "Net revenue": 93925, "Cost of sales": 43066, "Gross profit": 50859,
    "Operating profit": 11498, "Income before tax": 10244, "Income tax": 1949,
    "Net income (consolidated)": 8295, "Net income (attributable)": 8240,
    "D&A": 3451, "Cash from operations": 12087, "Cash from investing": -6879,
    "Capital spending": 4415, "Cash": 9159, "Short-term investments": 371,
    "Current assets": 27949, "Current liabilities": 32764, "Total assets": 107399,
    "Total liabilities": 86852, "PepsiCo equity": 20406, "Short-term debt": 6861,
    "Long-term debt": 42321, "Payables (combined line)": 25903, "Trade payables": 11704,
    "Supplier finance": 1681, "Interest expense (gross)": 1840,
}

validation = pd.DataFrame({
    "extracted": values_df[2025],
    "filing": pd.Series(filing_2025),
})
validation["difference"] = validation["extracted"] - validation["filing"]
validation["result"] = validation["difference"].apply(lambda d: "match" if d == 0 else "CHECK")
validation

,extracted,filing,difference,result
Net revenue,93925.0,93925,0.0,match
Cost of sales,43066.0,43066,0.0,match
Gross profit,50859.0,50859,0.0,match
Operating profit,11498.0,11498,0.0,match
Income before tax,10244.0,10244,0.0,match
Income tax,1949.0,1949,0.0,match
Net income (consolidated),8295.0,8295,0.0,match
Net income (attributable),8240.0,8240,0.0,match
D&A,3451.0,3451,0.0,match
Cash from operations,12087.0,12087,0.0,match


## Validation across all years

Every extracted value from 2017 to 2025 is compared with the figure printed in the most recent 10-K that reports that year. Statement figures come from the consolidated income statement, cash flow statement and balance sheet; trade payables from the note on accounts payable and other current liabilities; supplier finance obligations from the supplier finance program note. The 2017 balance sheet and trade payables are checked against the fiscal 2017 10-K. The table lists only the rows that do not match.

In [6]:
# Filing values, $ millions, as (value, 10-K year it was read from).
# Each year is taken from the most recent 10-K that reports it; 10-Ks are named by fiscal year. Capital spending is stored as a positive amount.
filing_values = {
    "Capital spending": {2017: (2969, 2019), 2018: (3282, 2020), 2019: (4232, 2021), 2020: (4240, 2022), 2021: (4625, 2023), 2022: (5207, 2024), 2023: (5518, 2025), 2024: (5318, 2025), 2025: (4415, 2025)},
    "Cash": {2017: (10610, 2017), 2018: (8721, 2019), 2019: (5509, 2020), 2020: (8185, 2021), 2021: (5596, 2022), 2022: (4954, 2023), 2023: (9711, 2024), 2024: (8505, 2025), 2025: (9159, 2025)},
    "Cash from investing": {2017: (-4403, 2019), 2018: (4564, 2020), 2019: (-6437, 2021), 2020: (-11619, 2022), 2021: (-3269, 2023), 2022: (-2430, 2024), 2023: (-5495, 2025), 2024: (-5472, 2025), 2025: (-6879, 2025)},
    "Cash from operations": {2017: (10030, 2019), 2018: (9415, 2020), 2019: (9649, 2021), 2020: (10613, 2022), 2021: (11616, 2023), 2022: (10811, 2024), 2023: (13442, 2025), 2024: (12507, 2025), 2025: (12087, 2025)},
    "Cost of sales": {2017: (28796, 2019), 2018: (29381, 2020), 2019: (30132, 2021), 2020: (31797, 2022), 2021: (37075, 2023), 2022: (40576, 2024), 2023: (41881, 2025), 2024: (41744, 2025), 2025: (43066, 2025)},
    "Current assets": {2017: (31027, 2017), 2018: (21893, 2019), 2019: (17645, 2020), 2020: (23001, 2021), 2021: (21783, 2022), 2022: (21539, 2023), 2023: (26950, 2024), 2024: (25826, 2025), 2025: (27949, 2025)},
    "Current liabilities": {2017: (20502, 2017), 2018: (22138, 2019), 2019: (20461, 2020), 2020: (23372, 2021), 2021: (26220, 2022), 2022: (26785, 2023), 2023: (31647, 2024), 2024: (31536, 2025), 2025: (32764, 2025)},
    "D&A": {2017: (2369, 2019), 2018: (2399, 2020), 2019: (2432, 2021), 2020: (2548, 2022), 2021: (2710, 2023), 2022: (2763, 2024), 2023: (2948, 2025), 2024: (3160, 2025), 2025: (3451, 2025)},
    "Gross profit": {2017: (34729, 2019), 2018: (35280, 2020), 2019: (37029, 2021), 2020: (38575, 2022), 2021: (42399, 2023), 2022: (45816, 2024), 2023: (49590, 2025), 2024: (50110, 2025), 2025: (50859, 2025)},
    "Income before tax": {2017: (9602, 2019), 2018: (9189, 2020), 2019: (9312, 2021), 2020: (9069, 2022), 2021: (9821, 2023), 2022: (10705, 2024), 2023: (11417, 2025), 2024: (11946, 2025), 2025: (10244, 2025)},
    "Income tax": {2017: (4694, 2019), 2018: (-3370, 2020), 2019: (1959, 2021), 2020: (1894, 2022), 2021: (2142, 2023), 2022: (1727, 2024), 2023: (2262, 2025), 2024: (2320, 2025), 2025: (1949, 2025)},
    "Long-term debt": {2017: (33796, 2017), 2018: (28295, 2019), 2019: (29148, 2020), 2020: (40370, 2021), 2021: (36026, 2022), 2022: (35657, 2023), 2023: (37595, 2024), 2024: (37224, 2025), 2025: (42321, 2025)},
    "Net income (attributable)": {2017: (4857, 2019), 2018: (12515, 2020), 2019: (7314, 2021), 2020: (7120, 2022), 2021: (7618, 2023), 2022: (8910, 2024), 2023: (9074, 2025), 2024: (9578, 2025), 2025: (8240, 2025)},
    "Net income (consolidated)": {2017: (4908, 2019), 2018: (12559, 2020), 2019: (7353, 2021), 2020: (7175, 2022), 2021: (7679, 2023), 2022: (8978, 2024), 2023: (9155, 2025), 2024: (9626, 2025), 2025: (8295, 2025)},
    "Net revenue": {2017: (63525, 2019), 2018: (64661, 2020), 2019: (67161, 2021), 2020: (70372, 2022), 2021: (79474, 2023), 2022: (86392, 2024), 2023: (91471, 2025), 2024: (91854, 2025), 2025: (93925, 2025)},
    "Operating profit": {2017: (10276, 2019), 2018: (10110, 2020), 2019: (10291, 2021), 2020: (10080, 2022), 2021: (11162, 2023), 2022: (11512, 2024), 2023: (11986, 2025), 2024: (12887, 2025), 2025: (11498, 2025)},
    "Payables (combined line)": {2017: (15017, 2017), 2018: (18112, 2019), 2019: (17541, 2020), 2020: (19592, 2021), 2021: (21159, 2022), 2022: (23371, 2023), 2023: (25137, 2024), 2024: (24454, 2025), 2025: (25903, 2025)},
    "PepsiCo equity": {2017: (11045, 2017), 2018: (14518, 2019), 2019: (14786, 2020), 2020: (13454, 2021), 2021: (16043, 2022), 2022: (17149, 2023), 2023: (18503, 2024), 2024: (18041, 2025), 2025: (20406, 2025)},
    "Short-term debt": {2017: (5485, 2017), 2018: (4026, 2019), 2019: (2920, 2020), 2020: (3780, 2021), 2021: (4308, 2022), 2022: (3414, 2023), 2023: (6510, 2024), 2024: (7082, 2025), 2025: (6861, 2025)},
    "Short-term investments": {2017: (8900, 2017), 2018: (272, 2019), 2019: (229, 2020), 2020: (1366, 2021), 2021: (392, 2022), 2022: (394, 2023), 2023: (292, 2024), 2024: (761, 2025), 2025: (371, 2025)},
    "Total assets": {2017: (79804, 2017), 2018: (77648, 2019), 2019: (78547, 2020), 2020: (92918, 2021), 2021: (92377, 2022), 2022: (92187, 2023), 2023: (100495, 2024), 2024: (99467, 2025), 2025: (107399, 2025)},
    "Total liabilities": {2017: (68823, 2017), 2018: (63046, 2019), 2019: (63679, 2020), 2020: (79366, 2021), 2021: (76226, 2022), 2022: (74914, 2023), 2023: (81858, 2024), 2024: (81296, 2025), 2025: (86852, 2025)},
    "Trade payables": {2017: (6727, 2017), 2018: (7213, 2019), 2019: (8013, 2020), 2020: (8853, 2021), 2021: (9834, 2022), 2022: (10732, 2023), 2023: (11635, 2024), 2024: (10997, 2025), 2025: (11704, 2025)},
    "Supplier finance": {2023: (1655, 2025), 2024: (1478, 2025), 2025: (1681, 2025)},
}

rows = []
for concept in values_df.index:
    if concept == "Interest expense (gross)":
        continue  # already entered from the filings
    for year in values_df.columns:
        extracted = values_df.loc[concept, year]
        if pd.isna(extracted):
            continue
        if year not in filing_values.get(concept, {}):
            rows.append({"concept": concept, "year": year, "extracted": extracted,
                         "filing": None, "source": "", "result": "awaiting 2018 10-K" if year == 2017 else "no filing value"})
            continue
        filed_value, source_year = filing_values[concept][year]
        rows.append({"concept": concept, "year": year, "extracted": extracted, "filing": filed_value,
                     "source": f"{source_year} 10-K",
                     "result": "match" if extracted == filed_value else "CHECK"})

full_validation = pd.DataFrame(rows)
print(full_validation["result"].value_counts().to_string())
full_validation[full_validation["result"] != "match"]

result
match    210


,concept,year,extracted,filing,source,result


### 2017 as first reported

The fiscal 2017 10-K reported some 2017 figures differently from the later 10-Ks used in this analysis, so 2017 is presented on the later, restated basis. Two presentation changes explain the differences.

**Pension income.** The operating profit difference of $233 million equals the 2017 "other pension and retiree medical benefits income" line in the fiscal 2019 10-K. From 2018, PepsiCo reports that income below operating profit and restated 2017 to match. The small cost of sales and gross profit differences ($11 million) are reclassifications of the same kind.

**Restricted cash.** Cash from operations is $36 million higher in the later filings. The only operating line that changed is "other, net", from $(341) million to $(305) million. The later filings reconcile cash flows to cash plus restricted cash rather than cash alone, so the change in restricted cash no longer runs through operating activities. The second table below derives restricted cash from the two cash totals reported for 2017; its increase matches the $36 million exactly.

Using the most recent filing keeps every year on the same presentation.

In [7]:
first_reported_2017 = {   # fiscal 2017 10-K, $ millions
    "Cost of sales": 28785, "Gross profit": 34740, "Operating profit": 10509,
    "Cash from operations": 9994,
}
display(pd.DataFrame({
    "first reported (FY2017 10-K)": pd.Series(first_reported_2017),
    "used here (latest 10-K)": values_df.loc[list(first_reported_2017), 2017],
}).assign(change=lambda t: t["used here (latest 10-K)"] - t["first reported (FY2017 10-K)"]))

# Restricted cash in 2017 = (cash + restricted cash, fiscal 2019 10-K) − (cash only, fiscal 2017 10-K)
restricted_cash = pd.DataFrame({
    "cash + restricted cash (FY2019 10-K)": {"start of 2017": 9169, "end of 2017": 10657},
    "cash only (FY2017 10-K)": {"start of 2017": 9158, "end of 2017": 10610},
})
restricted_cash["restricted cash"] = restricted_cash.iloc[:, 0] - restricted_cash.iloc[:, 1]
print("Increase in restricted cash during 2017:",
      restricted_cash.loc["end of 2017", "restricted cash"] - restricted_cash.loc["start of 2017", "restricted cash"])
restricted_cash

,first reported (FY2017 10-K),used here (latest 10-K),change
Cost of sales,28785,28796.0,11.0
Gross profit,34740,34729.0,-11.0
Operating profit,10509,10276.0,-233.0
Cash from operations,9994,10030.0,36.0


Increase in restricted cash during 2017: 36


,cash + restricted cash (FY2019 10-K),cash only (FY2017 10-K),restricted cash
start of 2017,9169,9158,11
end of 2017,10657,10610,47


## Segment data

Segment figures are reported only in the segment note of each 10-K, so they are entered from the filings. PepsiCo used a seven-segment structure (FLNA, QFNA, PBNA, LatAm, Europe, AMESA, APAC) through fiscal 2024 and a six-segment structure (PFNA, PBNA, IB Franchise, EMEA, LatAm Foods, Asia Pacific Foods) from fiscal 2025. The fiscal 2024 10-K still reports 2022–2024 on the seven-segment basis, and the fiscal 2025 10-K recasts 2023 and 2024 onto the six-segment basis. The seven-segment series therefore runs from 2017 to 2024, the six-segment series from 2023 to 2025, and 2023 and 2024 appear on both.

**Why earlier years are not restated onto six segments.** PepsiCo's current six-segment structure is not a simple regrouping of its earlier seven-segment structure. Although North America can be mapped cleanly (PFNA equals FLNA plus QFNA, and PBNA is unchanged, as the 2023 and 2024 figures reported on both structures confirm), the international segments were reorganised by both geography and type of business. PepsiCo did not publish the historical profit splits needed to restate 2017–2022 reliably. This analysis therefore keeps PepsiCo's reported seven-segment structure through 2024 and uses the reported six-segment structure for 2023–2025, with the overlapping years providing context for the transition. Creating a single six-segment series would require estimates based partly on revenue shares rounded to the nearest 5%, and would introduce false precision.

Segment revenue should equal consolidated net revenue. Segment operating profit does not include corporate unallocated expenses, so the check is division operating profit plus corporate unallocated expenses (a negative amount) against consolidated operating profit.

In [8]:
# Segment net revenue and operating profit, $ millions, as (revenue, operating profit).
# "Corporate" holds corporate unallocated expenses (operating profit only).
# Each year is read from the most recent 10-K that reports it on that structure.
segment_data = {
    "seven": {
        "FLNA": {2017: (15798, 4793), 2018: (16346, 5008), 2019: (17078, 5258), 2020: (18189, 5340), 2021: (19608, 5633), 2022: (23291, 6135), 2023: (24914, 6755), 2024: (24755, 6316)},
        "QFNA": {2017: (2503, 640), 2018: (2465, 637), 2019: (2482, 544), 2020: (2742, 669), 2021: (2751, 578), 2022: (3160, 604), 2023: (3101, 492), 2024: (2676, 303)},
        "PBNA": {2017: (20936, 2700), 2018: (21072, 2276), 2019: (21730, 2179), 2020: (22559, 1937), 2021: (25276, 2442), 2022: (26213, 5426), 2023: (27626, 2584), 2024: (27769, 2302)},
        "LatAm": {2017: (7208, 924), 2018: (7354, 1049), 2019: (7573, 1141), 2020: (6942, 1033), 2021: (8108, 1369), 2022: (9779, 1627), 2023: (11654, 2252), 2024: (11718, 2245)},
        "Europe": {2017: (10522, 1199), 2018: (10973, 1256), 2019: (11728, 1327), 2020: (11922, 1353), 2021: (13038, 1292), 2022: (12724, -1380), 2023: (13234, 767), 2024: (13874, 2019)},
        "AMESA": {2017: (3674, 789), 2018: (3657, 661), 2019: (3651, 671), 2020: (4573, 600), 2021: (6078, 858), 2022: (6438, 666), 2023: (6139, 807), 2024: (6217, 798)},
        "APAC": {2017: (2884, 401), 2018: (2794, 619), 2019: (2919, 477), 2020: (3445, 590), 2021: (4615, 673), 2022: (4787, 537), 2023: (4803, 713), 2024: (4845, 811)},
        "Corporate": {2017: -1170, 2018: -1396, 2019: -1306, 2020: -1442, 2021: -1683, 2022: -2103, 2023: -2384, 2024: -1907},
    },
    "six": {
        "PFNA": {2023: (28015, 7247), 2024: (27431, 6619), 2025: (27528, 6173)},
        "PBNA": {2023: (27626, 2584), 2024: (27769, 2302), 2025: (28197, 1089)},
        "IB Franchise": {2023: (4559, 567), 2024: (4879, 1462), 2025: (4997, 1769)},
        "EMEA": {2023: (16210, 1764), 2024: (16658, 1971), 2025: (18025, 2106)},
        "LatAm Foods": {2023: (10576, 1898), 2024: (10568, 2052), 2025: (10549, 2010)},
        "Asia Pacific Foods": {2023: (4485, 301), 2024: (4549, 377), 2025: (4629, 369)},
        "Corporate": {2023: -2375, 2024: -1896, 2025: -2018},
    },
}
segment_source = {   # fiscal year of the 10-K each year is read from
    "seven": {2017: 2019, 2018: 2020, 2019: 2021, 2020: 2022, 2021: 2023, 2022: 2024, 2023: 2024, 2024: 2024},
    "six": {2023: 2025, 2024: 2025, 2025: 2025},
}

rows = []
for structure, segments in segment_data.items():
    for segment, years in segments.items():
        for year, v in years.items():
            revenue, op = (0, v) if segment == "Corporate" else v
            rows.append({"structure": structure, "segment": segment, "year": year,
                         "revenue": revenue, "operating_profit": op,
                         "source": f"{segment_source[structure][year]} 10-K"})
segments_df = pd.DataFrame(rows)

# Reconciliation: segments sum to consolidated revenue; division profit + corporate = operating profit
recon = []
for (structure, year), grp in segments_df.groupby(["structure", "year"]):
    divisions = grp[grp["segment"] != "Corporate"]
    corporate = grp.loc[grp["segment"] == "Corporate", "operating_profit"].sum()
    recon.append({
        "structure": structure, "year": year,
        "segment revenue": divisions["revenue"].sum(),
        "consolidated revenue": values_df.loc["Net revenue", year],
        "division operating profit": divisions["operating_profit"].sum(),
        "corporate unallocated": corporate,
        "consolidated operating profit": values_df.loc["Operating profit", year],
    })
segment_recon = pd.DataFrame(recon)
segment_recon["revenue check"] = (segment_recon["segment revenue"] == segment_recon["consolidated revenue"]).map({True: "match", False: "CHECK"})
segment_recon["profit check"] = ((segment_recon["division operating profit"] + segment_recon["corporate unallocated"])
                                 == segment_recon["consolidated operating profit"]).map({True: "match", False: "CHECK"})
segment_recon

,structure,year,segment revenue,consolidated revenue,division operating profit,corporate unallocated,consolidated operating profit,revenue check,profit check
0,seven,2017,63525,63525.0,11446,-1170,10276.0,match,match
1,seven,2018,64661,64661.0,11506,-1396,10110.0,match,match
2,seven,2019,67161,67161.0,11597,-1306,10291.0,match,match
3,seven,2020,70372,70372.0,11522,-1442,10080.0,match,match
4,seven,2021,79474,79474.0,12845,-1683,11162.0,match,match
5,seven,2022,86392,86392.0,13615,-2103,11512.0,match,match
6,seven,2023,91471,91471.0,14370,-2384,11986.0,match,match
7,seven,2024,91854,91854.0,14794,-1907,12887.0,match,match
8,six,2023,91471,91471.0,14361,-2375,11986.0,match,match
9,six,2024,91854,91854.0,14783,-1896,12887.0,match,match


## Items below operating profit

Income before income taxes equals operating profit, plus other pension and retiree medical benefits income (or minus the expense), minus net interest expense and other. Both lines come from the income statement. The check below rebuilds income before taxes from these parts for every year. The 2017 and 2018 income statements showed interest expense and interest income separately; their net amount is used here.

In [9]:
# $ millions, from the income statement of the most recent 10-K reporting each year
pension_line = {2017: 233, 2018: 298, 2019: -44, 2020: 117, 2021: 522, 2022: 132, 2023: 250, 2024: -22, 2025: -133}
net_interest = {2017: 1151 - 244, 2018: 1525 - 306, 2019: 935, 2020: 1128, 2021: 1863, 2022: 939, 2023: 819, 2024: 919, 2025: 1121}

values_df.loc["Other pension and retiree medical (expense)/income"] = pd.Series(pension_line)
values_df.loc["Net interest expense and other"] = pd.Series(net_interest)
sources_df.loc["Other pension and retiree medical (expense)/income"] = "filing-note source: income statement, latest 10-K reporting the year"
sources_df.loc["Net interest expense and other"] = "filing-note source: income statement, latest 10-K reporting the year"

below_op = pd.DataFrame({
    "operating profit": values_df.loc["Operating profit"],
    "pension line": values_df.loc["Other pension and retiree medical (expense)/income"],
    "net interest and other": values_df.loc["Net interest expense and other"],
    "income before tax": values_df.loc["Income before tax"],
})
below_op["rebuilt"] = below_op["operating profit"] + below_op["pension line"] - below_op["net interest and other"]
below_op["result"] = (below_op["rebuilt"] == below_op["income before tax"]).map({True: "match", False: "CHECK"})
below_op

,operating profit,pension line,net interest and other,income before tax,rebuilt,result
2017,10276.0,233.0,907.0,9602.0,9602.0,match
2018,10110.0,298.0,1219.0,9189.0,9189.0,match
2019,10291.0,-44.0,935.0,9312.0,9312.0,match
2020,10080.0,117.0,1128.0,9069.0,9069.0,match
2021,11162.0,522.0,1863.0,9821.0,9821.0,match
2022,11512.0,132.0,939.0,10705.0,10705.0,match
2023,11986.0,250.0,819.0,11417.0,11417.0,match
2024,12887.0,-22.0,919.0,11946.0,11946.0,match
2025,11498.0,-133.0,1121.0,10244.0,10244.0,match


## Foreign exchange effect on revenue growth

Each MD&A reconciles reported net revenue growth to organic growth. The foreign exchange column shows how much currency translation moved reported growth, in percentage points as PepsiCo rounds them (to the nearest half point). Here a negative number means currency movements lowered reported growth; PepsiCo's own tables show the opposite sign, as the amount added back to reach organic growth.

In [10]:
fx_effect = pd.DataFrame({
    "reported revenue growth (%)": {2017: 1, 2018: 2, 2019: 4, 2020: 5, 2021: 13, 2022: 9, 2023: 6, 2024: 0, 2025: 2},
    "FX effect on reported growth (pp)": {2017: 0, 2018: -1, 2019: -2, 2020: -2, 2021: 1, 2022: -3, 2023: -2, 2024: -1.5, 2025: 0},
    "source": {2017: "2017 10-K", 2018: "2019 10-K", 2019: "2019 10-K", 2020: "2020 10-K", 2021: "2021 10-K",
               2022: "2022 10-K", 2023: "2023 10-K", 2024: "2024 10-K", 2025: "2025 10-K"},
})
fx_effect

,reported revenue growth (%),FX effect on reported growth (pp),source
2017,1,0.0,2017 10-K
2018,2,-1.0,2019 10-K
2019,4,-2.0,2019 10-K
2020,5,-2.0,2020 10-K
2021,13,1.0,2021 10-K
2022,9,-3.0,2022 10-K
2023,6,-2.0,2023 10-K
2024,0,-1.5,2024 10-K
2025,2,0.0,2025 10-K


## Normalisation items

The items below are the material one-off items identified in the filings. Pre-tax amounts affect the line shown; after-tax amounts are the effect on net income attributable to PepsiCo as stated in each filing. Positive amounts are charges (added back when normalising); negative amounts are gains or benefits (removed). Segment amounts are included only where the filing assigns the item to a segment. The 2025 Rockstar impairment has no separately disclosed after-tax amount: the filing gives $1,491 million after tax for all 2025 impairment and other charges of $1,946 million, of which Rockstar is $1,863 million.

In [11]:
normalisation_items = pd.DataFrame([
    {"year": 2017, "item": "Provisional net tax expense, TCJ Act", "line": "income tax",
     "pre_tax": 0, "after_tax": 2451, "segments": "none (tax item)", "source": "2019 10-K, cash flow statement; 2017 10-K, Note 5"},
    {"year": 2018, "item": "Other net tax benefits (international reorganisation $4,347m; tax audit conclusions $717m)", "line": "income tax",
     "pre_tax": 0, "after_tax": -5064, "segments": "none (tax item)", "source": "2019 10-K, items affecting comparability"},
    {"year": 2021, "item": "Charge related to cash tender offers", "line": "net interest expense and other",
     "pre_tax": 842, "after_tax": 677, "segments": "none (financing item)", "source": "2021 10-K, Note 8 and items affecting comparability"},
    {"year": 2022, "item": "Gain associated with the Juice Transaction", "line": "operating profit",
     "pre_tax": -3321, "after_tax": -2888, "segments": "PBNA -3,029; Europe -292", "source": "2022 10-K, Note 13 and items affecting comparability"},
    {"year": 2022, "item": "Russia-Ukraine conflict charges", "line": "operating profit",
     "pre_tax": 1415, "after_tax": 1124, "segments": "Europe 1,415", "source": "2022 10-K, impairment and other charges note"},
    {"year": 2025, "item": "Rockstar brand impairment", "line": "operating profit",
     "pre_tax": 1863, "after_tax": None, "segments": "PBNA 1,539; EMEA 251; IB Franchise 73", "source": "2025 10-K, impairment and other charges note"},
])
normalisation_items

,year,item,line,pre_tax,after_tax,segments,source
0,2017,"Provisional net tax expense, TCJ Act",income tax,0,2451.0,none (tax item),"2019 10-K, cash flow statement; 2017 10-K, Note 5"
1,2018,"Other net tax benefits (international reorganisation $4,347m; tax audit conclusions $717m)",income tax,0,-5064.0,none (tax item),"2019 10-K, items affecting comparability"
2,2021,Charge related to cash tender offers,net interest expense and other,842,677.0,none (financing item),"2021 10-K, Note 8 and items affecting comparability"
3,2022,Gain associated with the Juice Transaction,operating profit,-3321,-2888.0,"PBNA -3,029; Europe -292","2022 10-K, Note 13 and items affecting comparability"
4,2022,Russia-Ukraine conflict charges,operating profit,1415,1124.0,"Europe 1,415","2022 10-K, impairment and other charges note"
5,2025,Rockstar brand impairment,operating profit,1863,NaN,"PBNA 1,539; EMEA 251; IB Franchise 73","2025 10-K, impairment and other charges note"


## Recurring charges

PepsiCo reports two separate categories of charges that appear year after year rather than once. Each figure is the amount for that year alone, in $ millions before tax, taken from the cash flow statement of the most recent 10-K reporting the year. The figures are not cumulative.

**Restructuring and impairment charges** are the costs of PepsiCo's ongoing productivity programme, as described in the filings: closing or consolidating facilities, severance, and writing down assets that are no longer needed. They are reported in every year from 2017 to 2025.

**Impairment and other charges** are larger write-downs of brands, investments and other assets whose value has fallen below their carrying amount, together with related charges. The filings attribute most of each year's amount to a few items: the Russia-Ukraine conflict and brand portfolio decisions in 2022, SodaStream in 2023, TBG in 2024 and Rockstar in 2025. PepsiCo reports this category as a separate line only from 2022, so earlier years are left blank: a blank means the line did not exist, not that no such charges occurred.

Items already listed as normalisation items are included in these totals in the years they occur.

In [12]:
# $ millions, pre-tax, per year (not cumulative); None = no separate line reported that year
recurring_charges = pd.DataFrame({
    "restructuring and impairment charges": {2017: 295, 2018: 308, 2019: 370, 2020: 289, 2021: 247, 2022: 411, 2023: 445, 2024: 727, 2025: 983},
    "impairment and other charges": {2017: None, 2018: None, 2019: None, 2020: None, 2021: None, 2022: 3618, 2023: 1230, 2024: 714, 2025: 1946},
})
recurring_charges

,restructuring and impairment charges,impairment and other charges
2017,295,NaN
2018,308,NaN
2019,370,NaN
2020,289,NaN
2021,247,NaN
2022,411,3618.0
2023,445,1230.0
2024,727,714.0
2025,983,1946.0


## Export

The verified values, their sources, the full validation table, the segment data and its reconciliation, the foreign exchange effect, the normalisation items and the recurring charges are written to the project workbook. Existing sheets with the same names are replaced; any other sheets in the workbook are left untouched.

In [13]:
from pathlib import Path

workbook = Path("../analysis/PepsiCo_FSA.xlsx")
workbook.parent.mkdir(parents=True, exist_ok=True)

mode = "a" if workbook.exists() else "w"
extra = {"if_sheet_exists": "replace"} if mode == "a" else {}
try:
    with pd.ExcelWriter(workbook, engine="openpyxl", mode=mode, **extra) as writer:
        values_df.to_excel(writer, sheet_name="Raw Data")
        sources_df.to_excel(writer, sheet_name="Sources")
        full_validation.to_excel(writer, sheet_name="Validation", index=False)
        segments_df.to_excel(writer, sheet_name="Segment Data", index=False)
        segment_recon.to_excel(writer, sheet_name="Segment Reconciliation", index=False)
        fx_effect.to_excel(writer, sheet_name="FX Effect")
        normalisation_items.to_excel(writer, sheet_name="Normalisation Items", index=False)
        recurring_charges.to_excel(writer, sheet_name="Recurring Charges")
    print(f"Saved to {workbook.resolve()}")
except PermissionError:
    print("Could not save: close PepsiCo_FSA.xlsx in Excel (or wait for OneDrive to finish syncing), then run this cell again.")

Saved to C:\Users\fatih\OneDrive\Documents\My Portfolio Projects\Pepsico Analysis Series\analysis\PepsiCo_FSA.xlsx
